# Application RAG (Retrieval-Augmented Generation)
**Auteur :** Charkaoui Wissal  
**Description :** Assistant intelligent capable de répondre à des questions basées sur des documents PDF (Big Data, Power BI, JADE) en utilisant LangChain et Phi-2.

In [ ]:
# Installation des dépendances nécessaires
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers transformers accelerate bitsandbytes faiss-cpu pypdf gradio

In [1]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/RAG/data "
import os
os.makedirs(DATA_PATH, exist_ok=True)

print("Drive monté et dépendances installées")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive monté et dépendances installées


In [ ]:
# ================================================================
# LLM, EMBEDDINGS ET DOCUMENTS (LANGCHAIN)
# ================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline


# ---------------------------------------------------------
# 1. GESTION DU MATÉRIEL (DEVICE)
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé: {device}")

# ---------------------------------------------------------
# 2. CONFIGURATION DU MODÈLE : Microsoft Phi-2 (2.7B)
# ---------------------------------------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc

# Nettoyage préventif de la mémoire vive et vidéo
gc.collect()
torch.cuda.empty_cache()

# Identification du modèle stable choisi pour le RAG
model_name = "microsoft/phi-2"

print(f" Chargement du modèle {model_name}...")

# Initialisation du Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement des poids du modèle (Précision float16 si GPU disponible)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# Allocation sur le processeur graphique (GPU)
if torch.cuda.is_available():
    model = model.to("cuda")
    print(" Modèle chargé sur GPU")
else:
    print(" Modèle chargé sur CPU")

# Configuration des paramètres de génération du texte
print(" Configuration du pipeline...")
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.85,
    top_k=50,
    repetition_penalty=1.2,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id
)

# Intégration du pipeline dans l'écosystème LangChain
llm = HuggingFacePipeline(pipeline=generation_pipeline)

print(f" Phi-2 chargé avec succès !")
print(f" Taille du modèle : 2.7B paramètres")
print(f" Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else "")

# ---------------------------------------------------------
# 3. EMBEDDINGS (Modèle de représentation vectorielle)
# ---------------------------------------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("Embeddings chargés")

# ---------------------------------------------------------
# 4. CHARGEMENT DES DOCUMENTS PDF
# ---------------------------------------------------------
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()

print(f"{len(documents)} documents chargés")

# ---------------------------------------------------------
# 5. DÉCOUPAGE DES TEXTES (SPLITTING)
# ---------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

splits = text_splitter.split_documents(documents)

print(f"{len(splits)} chunks créés")

Device utilisé: cuda
 Chargement du modèle microsoft/phi-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


 Modèle chargé sur GPU
 Configuration du pipeline...
 Phi-2 chargé avec succès !
 Taille du modèle : 2.7B paramètres
 Mémoire GPU utilisée : 5.56 GB


/tmp/ipython-input-1091797649.py:76: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generation_pipeline)
/tmp/ipython-input-1091797649.py:85: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embeddings chargés
235 documents chargés
246 chunks créés


In [24]:
!ls "/content/drive/MyDrive/RAG/data "

'CH2_Big Data Analytics.pdf'  'formation power BI perf.pdf'
'CH3_Big Data Analytics.pdf'   Présentation_SMA_JADE.pdf
 chap3.pdf


In [3]:
import langchain
import langchain_community
import numpy
import torch

print("LangChain version:", langchain.__version__)
print("LangChain-Community version:", langchain_community.__version__)
print("NumPy version:", numpy.__version__)
print("Torch version:", torch.__version__)


LangChain version: 1.2.0
LangChain-Community version: 0.4.1
NumPy version: 2.3.5
Torch version: 2.9.1+cu128


In [ ]:
# ================================================================
# CELLULE 3 - CHAÎNE RAG ET INTERFACE UTILISATEUR (GRADIO)
# Développement et Optimisation : Charkaoui Wissal
# ================================================================

import gradio as gr
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores.faiss import FAISS

# ---------------------------------------------------------
# 1. BASE DE DONNÉES VECTORIELLE (FAISS)
# ---------------------------------------------------------
# Création de l'index à partir des segments de texte et des embeddings
vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

# Configuration du moteur de recherche (Top 5 documents les plus proches)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("✅ Index FAISS créé avec succès")

# ---------------------------------------------------------
# 2. CONSTRUCTION DE LA CHAÎNE RAG (LangChain Expression Language)
# ---------------------------------------------------------

# Définition du comportement de l'assistant (Prompt Engineering)
template = """
Tu es un assistant expert qui répond aux questions en te basant sur les documents fournis.
Utilise les informations du contexte ci-dessous pour répondre à la question.
Si tu ne trouves pas la réponse dans le contexte, dis-le clairement.
Réponds de manière concise et précise, en français uniquement.

Contexte : {context}

Question : {question}

Réponse détaillée en français :"""

prompt = ChatPromptTemplate.from_template(template)

# --- FONCTIONS UTILITAIRES DE LA CHAÎNE ---

def format_docs(docs):
    """Concatène le contenu des documents récupérés."""
    return "\n\n".join(doc.page_content for doc in docs)

def call_llm(prompt_value):
    """
    Gestionnaire d'appel au modèle : convertit les entrées en texte 
    et extrait proprement la réponse générée par Phi-2.
    """
    if hasattr(prompt_value, 'to_string'):
        prompt_str = prompt_value.to_string()
    elif hasattr(prompt_value, 'text'):
        prompt_str = prompt_value.text
    else:
        prompt_str = str(prompt_value)

    response = llm.invoke(prompt_str)

    # Extraction sécurisée selon le format de sortie (str, dict ou list)
    if isinstance(response, str):
        return response
    elif isinstance(response, dict):
        for key in ['generated_text', 'text', 'content']:
            if key in response: return response[key]
        return str(list(response.values())[0]) if response else ""
    elif isinstance(response, list) and len(response) > 0:
        first = response[0]
        if isinstance(first, dict):
            for key in ['generated_text', 'text', 'content']:
                if key in first: return first[key]
        return str(first)
    elif hasattr(response, 'content'):
        return response.content
    else:
        return str(response)

def retrieve_docs(inputs):
    """
    Récupère les documents pertinents en utilisant l'API 'invoke'
    pour une compatibilité maximale avec LangChain.
    """
    query = inputs['question'] if isinstance(inputs, dict) and 'question' in inputs else str(inputs)
    return retriever.invoke(query)

# --- ASSEMBLAGE DE LA CHAÎNE ---
rag_chain = (
    {
        "context": RunnableLambda(retrieve_docs) | format_docs,
        "question": lambda x: x["question"] if isinstance(x, dict) else x
    }
    | prompt
    | call_llm
)

print("✅ Chaîne RAG opérationnelle")

# ---------------------------------------------------------
# 3. LOGIQUE DE L'INTERFACE DE CHAT
# ---------------------------------------------------------

def chat_interface(query):
    """Fonction principale traitant la question, la réponse et les sources."""
    if not query.strip():
        return "Veuillez poser une question."

    try:
        # Exécution de la chaîne RAG
        answer = rag_chain.invoke({"question": query})

        # Nettoyage de la réponse pour supprimer le texte du prompt
        if isinstance(answer, str):
            for marker in ["Réponse détaillée en français :", "Réponse :", "Answer:"]:
                if marker in answer:
                    answer = answer.split(marker)[-1].strip()
                    break
            if query in answer:
                answer = answer.replace(query, "").strip()

        # Récupération des métadonnées pour l'affichage des sources
        docs = retriever.invoke(query)
        sources_text = "\n\n### 📚 Sources utilisées\n"
        seen_sources = set()

        for doc in docs:
            source = doc.metadata.get('source', 'Document inconnu').split('/')[-1]
            if source not in seen_sources:
                seen_sources.add(source)
                page = doc.metadata.get('page', 'N/A')
                sources_text += f"{len(seen_sources)}. **{source}** (page {page})\n"

        return f"### 💬 Réponse\n\n{answer}\n{sources_text}"

    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"❌ **Une erreur est survenue**\n\n```\n{str(e)}\n```"

# ---------------------------------------------------------
# 4. INTERFACE GRAPHIQUE (GRADIO)
# ---------------------------------------------------------

with gr.Blocks(title="RAG Application - Charkaoui Wissal", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🤖 Assistant RAG Intelligent")
    gr.Markdown("### 💡 Analyse de documents PDF (Power BI, Big Data, Framework JADE...)")

    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(
                label="❓ Votre question",
                placeholder="Exemple : Quelle est la différence entre SUM et SUMX en DAX ?",
                lines=4
            )

            with gr.Row():
                submit = gr.Button("🚀 Générer la réponse", variant="primary", size="lg")
                clear = gr.Button("🗑️ Effacer", size="lg")

            gr.Examples(
                examples=[
                    "Quelle est la différence entre SUM et SUMX en DAX ?",
                    "Qu'est-ce que Big Data Analytics ?",
                    "Explique-moi le framework JADE",
                    "Comment fonctionne Power BI ?",
                ],
                inputs=question
            )

    output = gr.Markdown(label="📝 Réponse")

    # Événements
    submit.click(fn=chat_interface, inputs=question, outputs=output)
    clear.click(fn=lambda: ("", ""), inputs=None, outputs=[question, output])

print("✅ Interface prête pour le lancement")
app.launch(debug=True, share=True)

FAISS index créé
RAG chain prête


/tmp/ipython-input-2780766016.py:164: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="RAG LangChain avec Mistral-7B", theme=gr.themes.Soft()) as app:


✅ Interface Gradio prête
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://497b9a35c1b1342af2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://497b9a35c1b1342af2.gradio.live


In [30]:
!pip install -q langchain-huggingface